# Import

In [4]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [5]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [6]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [7]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [8]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [9]:
for c in communities:
    print(len(c))

4080
3322
2931
2908
2055
1673
1039
1024
960
586
408
224
185
41
41
37
36
31
25
21
21
16
13
13
12
11
10
9
7
7
7
7
7
7
6
6
6
5
5
5
5
5
4
4
4
4
4
4
3
3
3
3
2
2
2
2
2
2
2
2
2
2
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


## Helpful functions (big object, drop NAN)

In [10]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [11]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [12]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{RESULT_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['56061', '6421', '23215', '26135', '9898', '23484', '26292', '27300', '9987', '8073', '6767', '10923', '8780', '8880', '23039', '4076', '27309', '2794', '29896', '5814', '57223', '7756', '5955', '55858', '10330', '10116', '28969', '5110', '23518', '10625', '81688', '9419', '9019', '23741', '9325', '7572', '54542', '55973', '53635', '6782', '130074', '8087', '9736', '10963', '11177', '2764', '55833', '10049', '5935', '51603', '10153', '285672', '84901', '8028', '22850', '10527', '64225', '9689', '10522', '30836', '4673', '55031', '7594', '55556', '25843', '27000', '57559', '10473', '8882', '8624', '51029', '2029', '54765', '10428', '9991', '51042', '22838', '9470', '26225', '5813', '9733', '26156', '55661', '7705', '51115', '10301', '10492', '81627', '23648', '6651', '92906', '79647', '8872', '387263', '4154', '9643', '7803', '7072', '573', '3301', '51430', '54165', '11235', '7873', '8099', '6059', '3267', '9802', '64318', '55884', '79073', '10360', '8634', '5876', '3621', '25948', '2

In [13]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{RESULT_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['16', '47', '50', '52', '97', '132', '143', '158', '159', '204', '205', '262', '271', '291', '292', '293', '310', '318', '327', '353', '439', '471', '475', '498', '506', '509', '513', '514', '515', '516', '517', '518', '520', '521', '522', '523', '526', '527', '528', '529', '533', '534', '537', '539', '550', '573', '617', '642', '689', '690', '705', '708', '740', '741', '746', '754', '790', '811', '819', '821', '833', '835', '889', '900', '908', '955', '984', '988', '989', '997', '1024', '1039', '1054', '1068', '1070', '1105', '1155', '1196', '1198', '1201', '1207', '1209', '1327', '1329', '1337', '1339', '1340', '1345', '1346', '1347', '1349', '1350', '1351', '1352', '1353', '1355', '1416', '1427', '1429', '1431', '1456', '1459', '1460', '1477', '1478', '1479', '1503', '1537', '1615', '1629', '1633', '1635', '1650', '1653', '1655', '1656', '1657', '1659', '1660', '1662', '1665', '1678', '1716', '1723', '1725', '1736', '1737', '1738', '1743', '1787', '1797', '1801', '1802', '1819', '

## NCBI to HGNC

In [14]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [15]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [16]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [17]:
print(len(COMMUNITIES_HGNC))

13


In [18]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 1 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 1 NaN entries
Community 5: dropped 1 NaN entries
Community 6: dropped 5 NaN entries
Community 7: dropped 2 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries

Total dropped across all communities: 10
Community 0: dropped 2 NaN entries
Community 1: dropped 3 NaN entries
Community 2: dropped 9 NaN entries
Community 3: dropped 4 NaN entries
Community 4: dropped 2 NaN entries
Community 5: dropped 5 NaN entries
Community 6: dropped 7 NaN entries
Community 7: dropped 5 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Commun

In [19]:
with open(f"{RESULT_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{RESULT_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [20]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

13
178


# Categoization Prep

### GO-slim

In [21]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [22]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [23]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [24]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [25]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [26]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [27]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [28]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [29]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

In [30]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Adjusted P-value'], ascending=True)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [31]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [32]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [33]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 1097
Number of filtered terms: 124
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_48708\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
2133,0,RNA Binding (GO:0003723),351/1411,6.843116e-143,[binding]
0,0,"mRNA Splicing, Via Spliceosome (GO:0000398)",123/211,2.514856e-96,[cellular process]
1,0,"RNA Splicing, Via Transesterification Reactions With Bulged Adenosine As Nucleophile (GO:0000377)",109/180,9.145320e-88,[cellular process]
2,0,mRNA Processing (GO:0006397),117/214,2.107879e-87,[cellular process]
3,0,RNA Processing (GO:0006396),85/183,4.376312e-55,[cellular process]
2541,0,Nucleus (GO:0005634),472/4487,6.410286e-53,[cellular anatomical structure]
4,0,RNA Splicing (GO:0008380),58/98,4.438204e-45,[cellular process]
2543,0,U2-type Spliceosomal Complex (GO:0005684),52/90,3.123148e-40,[protein-containing complex]
2544,0,Spliceosomal snRNP Complex (GO:0097525),40/57,6.119275e-36,[protein-containing complex]
2134,0,mRNA Binding (GO:0003729),82/282,4.876571e-35,[binding]


Size of community: 628
Number of filtered terms: 4
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
2312,1,Actin Cytoskeleton (GO:0015629),35/327,4.779277e-08,[cellular anatomical structure]
0,1,Actin Filament Bundle Organization (GO:0061572),10/33,8.269470e-05,[cellular process]
1,1,Actin Filament Organization (GO:0007015),19/144,1.300572e-04,[cellular process]
2,1,Actin Filament Bundle Assembly (GO:0051017),9/33,3.628275e-04,[cellular process]


...

Size of community: 214
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
21,10,Olfactory Receptor Activity (GO:0004984),169/362,1.410727e-268,[molecular transducer activity]
0,10,Sensory Perception Of Smell (GO:0007608),104/230,2.754928e-150,[multicellular organismal process]
1,10,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),62/141,1.501597e-85,[response to stimulus]
2,10,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),61/139,3.251653e-84,[response to stimulus]
3,10,Sensory Perception Of Chemical Stimulus (GO:0007606),48/110,1.144249e-65,[multicellular organismal process]


Size of community: 132
Number of filtered terms: 15
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Intermediate Filament Organization (GO:0045109),37/68,1.590339e-62,[cellular process]
121,11,Cornified Envelope (GO:0001533),23/41,3.767917e-39,[cellular anatomical structure]
122,11,Keratin Filament (GO:0045095),22/39,8.680165e-38,[cellular anatomical structure]
1,11,Supramolecular Fiber Organization (GO:0097435),38/316,1.015771e-35,[cellular process]
123,11,Intermediate Filament (GO:0005882),23/69,6.461525e-33,[cellular anatomical structure]
2,11,Epidermis Development (GO:0008544),20/85,1.870174e-24,[developmental process]
3,11,Epithelium Development (GO:0060429),21/154,1.446252e-20,[developmental process]
4,11,Epithelial Cell Differentiation (GO:0030855),20/132,1.446252e-20,"[developmental process, cellular process]"
5,11,Keratinocyte Differentiation (GO:0030216),11/42,4.221335e-14,"[developmental process, cellular process]"
6,11,Skin Development (GO:0043588),12/68,3.243378e-13,[developmental process]


11 out of 13 communities had significant GO terms.


In [34]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,RNA Binding (GO:0003723),351/1411,6.843116e-143,[binding],GO_Molecular_Function_2023,1.677234e-145,0.0,0.0,7.920093,2640.224035,EIF4A1;TCERG1;RTCA;RRP1;PPAN;HNRNPU;GPATCH4;HN...,0.248760
1,0,1097,"mRNA Splicing, Via Spliceosome (GO:0000398)",123/211,2.514856e-96,[cellular process],GO_Biological_Process_2023,1.179023e-99,0.0,0.0,27.000245,6150.419243,GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRPD2;SN...,0.582938
2,0,1097,"RNA Splicing, Via Transesterification Reaction...",109/180,9.145320e-88,[cellular process],GO_Biological_Process_2023,8.575078e-91,0.0,0.0,29.262246,6068.591273,HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRPD2;RBMX2;SNR...,0.605556
3,0,1097,mRNA Processing (GO:0006397),117/214,2.107879e-87,[cellular process],GO_Biological_Process_2023,2.964668e-90,0.0,0.0,23.146455,4771.546614,TCERG1;GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SN...,0.546729
4,0,1097,RNA Processing (GO:0006396),85/183,4.376312e-55,[cellular process],GO_Biological_Process_2023,8.206867e-58,0.0,0.0,16.117055,2118.505656,TCERG1;FASTKD1;GEMIN2;HNRNPU;HNRNPR;TENT4A;USP...,0.464481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2001,11,132,Epidermal Cell Differentiation (GO:0009913),11/54,6.623089e-13,"[developmental process, cellular process]",GO_Biological_Process_2023,5.094684e-14,0.0,0.0,41.913319,1282.882613,TGM1;SPRR2F;SPRR3;SPRR2G;CDSN;KRT16;LORICRIN;K...,0.203704
2002,11,132,Peptide Cross-Linking (GO:0018149),9/27,8.824325e-13,[cellular process],GO_Biological_Process_2023,7.636435e-14,0.0,0.0,80.691057,2437.133003,SPRR2E;TGM1;KRT2;KRT1;LORICRIN;KRT10;TGM5;IVL;...,0.333333
2003,11,132,Desmosome (GO:0030057),6/17,4.726332e-09,[cellular anatomical structure],GO_Cellular_Component_2023,8.593331e-10,0.0,0.0,85.961039,1794.425035,CDSN;KAZN;DSG1;DSG3;DSG4;DSC1,0.352941
2004,11,132,Intermediate Filament Cytoskeleton (GO:0045111),9/81,1.386349e-08,[cellular anatomical structure],GO_Cellular_Component_2023,3.150793e-09,0.0,0.0,20.117886,393.819930,KRT80;KRT3;KRT16;KRT2;KRT36;KRT10;KRT20;KRT75;...,0.111111


In [35]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,"[AARSD1, ACIN1, ADNP2, AGFG1, AK2, AKAP1, AKIR...",724,373
1,1,628,"[AHNAK, AIF1L, AUTS2, BIN3, CALD1, CDC42EP3, C...",43,585
2,2,887,"[ACOD1, ACTA2, ACTG2, ADAM12, ADAM19, ADAM8, A...",679,208
3,3,1036,"[ACKR1, ACKR3, ACKR4, ADAP2, ADCYAP1, ADCYAP1R...",612,424
4,4,824,"[ABCA1, ABCA5, ABCA6, ABCA8, ABCB1, ABCB4, ABC...",348,476
5,5,696,"[ABRAXAS1, ACTL6A, AHCTF1, AJUBA, ANLN, ANP32E...",525,171
6,6,508,"[ANKS3, ARL13B, ARL3, ARMC2, B9D1, BBIP1, BBOF...",135,373
7,7,323,[],0,323
8,8,393,"[ABI2, ACTN1, AKAP13, AKT1, AKT2, AKT3, ARAF, ...",361,32
9,9,319,"[AAK1, ACBD3, ALS2, ANK2, ANK3, ANKRD27, AP1G2...",290,29


### KEGG

In [36]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [37]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1097
Number of filtered terms: 5
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_48708\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,Spliceosome,84/150,6.064668e-64,[Transcription]
1,0,RNA transport,39/186,1.624623e-11,[]
2,0,mRNA surveillance pathway,25/98,2.302962e-09,[Translation]
3,0,Ubiquitin mediated proteolysis,29/140,1.197541e-08,"[Folding, sorting and degradation]"
4,0,Ribosome biogenesis in eukaryotes,20/108,3.104751e-05,[Translation]


Size of community: 887
Number of filtered terms: 61
Number of unmapped terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Cytokine-cytokine receptor interaction,100/295,7.514726e-59,[Signaling molecules and interaction]
1,2,Hematopoietic cell lineage,43/99,4.611095e-30,[Immune system]
2,2,Viral protein interaction with cytokine and cytokine receptor,43/100,5.170064e-30,[Signaling molecules and interaction]
3,2,Rheumatoid arthritis,41/93,2.995129e-29,[Immune disease]
4,2,Amoebiasis,39/102,4.618380e-25,[Infectious disease: parasitic]
5,2,Osteoclast differentiation,40/127,4.367218e-22,[Development and regeneration]
6,2,Inflammatory bowel disease,27/65,1.501708e-18,[Immune disease]
7,2,Cell adhesion molecules,39/148,1.501708e-18,[]
8,2,JAK-STAT signaling pathway,40/162,5.759437e-18,[Signal transduction]
9,2,ECM-receptor interaction,30/88,7.499043e-18,[Signaling molecules and interaction]


...

Size of community: 214
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Olfactory transduction,204/440,0.0,[Sensory system]


Size of community: 132
Number of filtered terms: 2
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Staphylococcus aureus infection,20/95,1.870074e-24,[Infectious disease: bacterial]
1,11,Estrogen signaling pathway,19/137,6.722506e-20,[Endocrine system]


9 out of 13 communities had significant GO terms.


In [38]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,Spliceosome,84/150,6.064668e-64,[Transcription],KEGG_2021_Human,5.831412e-66,0.0,0.0,23.666697,3554.911944,TCERG1;RBM25;DDX46;DDX42;HNRNPU;USP39;PQBP1;SN...,0.560000
1,0,1097,RNA transport,39/186,1.624623e-11,[],KEGG_2021_Human,3.124275e-13,0.0,0.0,4.703291,135.428454,EIF4A1;NXT1;RBM8A;GEMIN2;DDX20;PHAX;NMD3;CASC3...,0.209677
2,0,1097,mRNA surveillance pathway,25/98,2.302962e-09,[Translation],KEGG_2021_Human,6.643159e-11,0.0,0.0,6.015513,140.972640,DAZAP1;NXT1;RBM8A;PPP2R2A;CASC3;SMG7;GSPT1;SMG...,0.255102
3,0,1097,Ubiquitin mediated proteolysis,29/140,1.197541e-08,"[Folding, sorting and degradation]",KEGG_2021_Human,4.605926e-10,0.0,0.0,4.597024,98.829154,CUL5;UBA6;MGRN1;CUL3;UBE2D3;CUL2;UBE3A;UBE3B;R...,0.207143
4,0,1097,Ribosome biogenesis in eukaryotes,20/108,3.104751e-05,[Translation],KEGG_2021_Human,1.492669e-06,0.0,0.0,3.970414,53.262891,NOP56;NXT1;HEATR1;WDR75;NMD3;SNU13;PWP2;GNL2;U...,0.185185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,9,319,Vasopressin-regulated water reabsorption,11/44,1.144601e-09,[Excretory system],KEGG_2021_Human,6.867604e-11,0.0,0.0,21.264069,497.613684,NSF;DYNC1LI1;RAB5B;DYNC1LI2;RAB5C;STX4;DYNLL2;...,0.250000
260,9,319,Lysosome,14/128,2.201653e-07,[Transport and catabolism],KEGG_2021_Human,1.761323e-08,0.0,0.0,7.878573,140.668902,CTSZ;AP4E1;CLTB;AP3B1;IGF2R;AP4M1;GGA2;GGA1;GG...,0.109375
261,10,214,Olfactory transduction,204/440,0.000000e+00,[Sensory system],KEGG_2021_Human,0.000000e+00,0.0,0.0,1689.915254,inf,OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17...,0.463636
262,11,132,Staphylococcus aureus infection,20/95,1.870074e-24,[Infectious disease: bacterial],KEGG_2021_Human,6.233579e-25,0.0,0.0,47.126190,2626.562989,KRT24;KRT35;KRT34;KRT12;KRT32;KRT10;KRT20;KRT3...,0.210526


In [39]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,"[ACIN1, BCAS2, BUD31, CASC3, CDC40, CDC5L, CHE...",168,929
1,1,628,[],0,628
2,2,887,"[ACTA2, ANGPT1, ANGPT2, ANPEP, ANTXR1, ANTXR2,...",389,498
3,3,1036,"[ACKR3, ACKR4, ADCYAP1, ADCYAP1R1, ADORA2A, AD...",188,848
4,4,824,"[ABCA1, ABCA5, ABCA6, ABCA8, ABCB1, ABCB4, ABC...",244,580
5,5,696,"[ABRAXAS1, ATM, ATR, AURKA, BABAM1, BABAM2, BA...",192,504
6,6,508,[],0,508
7,7,323,[],0,323
8,8,393,"[ABI2, ACTN1, AKAP13, AKT1, AKT2, AKT3, ARAF, ...",335,58
9,9,319,"[AP1G2, AP1S2, AP3B1, AP3S1, AP4E1, AP4M1, AP4...",76,243


### Reactome

In [40]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [41]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 1097
Number of filtered terms: 20
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_48708\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,mRNA Splicing R-HSA-72172,109/189,7.202324e-85,[Metabolism of RNA]
1,0,Processing Of Capped Intron-Containing Pre-mRNA R-HSA-72203,121/242,7.481560e-85,[Metabolism of RNA]
2,0,mRNA Splicing - Major Pathway R-HSA-72163,104/181,4.424829e-81,[Metabolism of RNA]
3,0,Metabolism Of RNA R-HSA-8953854,184/666,2.603743e-78,[Metabolism of RNA]
4,0,mRNA Splicing - Minor Pathway R-HSA-72165,26/49,1.886502e-18,[Metabolism of RNA]
5,0,RNA Polymerase II Transcription Termination R-HSA-73856,29/67,1.471935e-17,[Gene expression (Transcription)]
6,0,Antigen Processing: Ubiquitination And Proteasome Degradation R-HSA-983168,58/307,5.177237e-15,[Immune System]
7,0,rRNA Processing In Nucleus And Cytosol R-HSA-8868773,44/189,8.716226e-15,[Metabolism of RNA]
8,0,Major Pathway Of rRNA Processing In Nucleolus And Cytosol R-HSA-6791226,42/179,2.813048e-14,[Metabolism of RNA]
9,0,mRNA 3-End Processing R-HSA-72187,24/58,4.031876e-14,[Metabolism of RNA]


Size of community: 887
Number of filtered terms: 65
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Immune System R-HSA-168256,322/1943,1.137758e-105,[Immune System]
1,2,Extracellular Matrix Organization R-HSA-1474244,118/291,2.794607e-80,[Extracellular matrix organization]
2,2,Cytokine Signaling In Immune System R-HSA-1280215,168/702,3.965941e-75,[Immune System]
3,2,Collagen Formation R-HSA-1474290,53/90,3.206071e-46,[Extracellular matrix organization]
4,2,Signaling By Interleukins R-HSA-449147,104/453,1.924853e-43,[Immune System]
5,2,Assembly Of Collagen Fibrils And Other Multimeric Structures R-HSA-2022090,42/57,6.355467e-43,[Extracellular matrix organization]
6,2,Immunoregulatory Interactions Between A Lymphoid And A non-Lymphoid Cell R-HSA-198933,50/123,1.934635e-33,[Immune System]
7,2,Innate Immune System R-HSA-168249,135/1035,1.043046e-28,[Immune System]
8,2,Degradation Of Extracellular Matrix R-HSA-1474228,43/109,4.151957e-28,[Extracellular matrix organization]
9,2,Interferon Alpha/Beta Signaling R-HSA-909733,36/72,5.331415e-28,[Immune System]


...

Size of community: 214
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Olfactory Signaling Pathway R-HSA-381753,199/401,0.000000e+00,[Sensory Perception]
1,10,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,199/393,0.000000e+00,[Sensory Perception]
2,10,Sensory Perception R-HSA-9709957,199/616,2.281963e-294,[Sensory Perception]


Size of community: 132
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Keratinization R-HSA-6805567,124/208,2.425853e-253,[Developmental Biology]
1,11,Developmental Biology R-HSA-1266738,124/1073,4.973656e-148,[Developmental Biology]
2,11,Formation Of Cornified Envelope R-HSA-6809371,40/74,1.055509e-68,[Developmental Biology]


10 out of 13 communities had significant GO terms.


In [42]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,mRNA Splicing R-HSA-72172,109/189,7.202324e-85,[Metabolism of RNA],Reactome_2022,1.640620e-87,0.0,0.0,25.957831,5.187149e+03,HNRNPU;HNRNPR;CASC3;CWC27;CCAR1;SNRPD2;MAGOH;M...,0.576720
1,0,1097,Processing Of Capped Intron-Containing Pre-mRN...,121/242,7.481560e-85,[Metabolism of RNA],Reactome_2022,3.408456e-87,0.0,0.0,19.243852,3.831425e+03,HNRNPU;HNRNPR;CASC3;CWC27;CCAR1;SNRPD2;MAGOH;M...,0.500000
2,0,1097,mRNA Splicing - Major Pathway R-HSA-72163,104/181,4.424829e-81,[Metabolism of RNA],Reactome_2022,3.023801e-83,0.0,0.0,25.606571,4.865455e+03,HNRNPU;HNRNPR;CASC3;CWC27;CCAR1;SNRPD2;MAGOH;M...,0.574586
3,0,1097,Metabolism Of RNA R-HSA-8953854,184/666,2.603743e-78,[Metabolism of RNA],Reactome_2022,2.372431e-80,0.0,0.0,7.702172,1.412138e+03,LTV1;EIF4A1;GEMIN2;SPPL2A;HNRNPU;HNRNPR;PHAX;P...,0.276276
4,0,1097,mRNA Splicing - Minor Pathway R-HSA-72165,26/49,1.886502e-18,[Metabolism of RNA],Reactome_2022,2.148636e-20,0.0,0.0,19.927739,9.024649e+02,SF3B4;SF3B5;SF3B3;DDX23;SRSF1;DDX42;YBX1;RNPC3...,0.530612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,10,214,Expression And Translocation Of Olfactory Rece...,199/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,1339.796564,inf,OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17...,0.506361
840,10,214,Sensory Perception R-HSA-9709957,199/616,2.281963e-294,[Sensory Perception],Reactome_2022,2.281963e-294,0.0,0.0,616.215987,4.166452e+05,OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;OR4K17...,0.323052
841,11,132,Keratinization R-HSA-6805567,124/208,2.425853e-253,[Developmental Biology],Reactome_2022,7.580792e-255,0.0,0.0,3650.619048,2.136100e+06,KRTAP24-1;LCE1A;TGM1;CAPNS1;LIPM;KRTAP3-2;KRTA...,0.596154
842,11,132,Developmental Biology R-HSA-1266738,124/1073,4.973656e-148,[Developmental Biology],Reactome_2022,3.108535e-149,0.0,0.0,309.003688,1.056641e+05,KRTAP24-1;LCE1A;TGM1;CAPNS1;LIPM;KRTAP3-2;KRTA...,0.115564


In [43]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1097,"[ANP32A, ARIH2, ASB1, BCAS2, BLMH, BOP1, BTBD1...",276,821
1,1,628,[],0,628
2,2,887,"[ACTA2, ACTG2, ADA2, ADAM12, ADAM19, ADAM8, AD...",493,394
3,3,1036,"[ABCC8, ACKR1, ACKR2, ACKR3, ACKR4, ADCYAP1, A...",207,829
4,4,824,"[AASS, ABCA1, ABCA5, ABCA6, ABCA8, ABCB1, ABCB...",316,508
5,5,696,"[ABRAXAS1, ACTL6A, AEBP2, AHCTF1, AJUBA, ANLN,...",466,230
6,6,508,"[ARL13B, ARL3, B9D1, BBIP1, BBS1, BBS2, BBS4, ...",35,473
7,7,323,[],0,323
8,8,393,"[ABI2, ABR, ACTN1, AKAP13, AKT1, AKT2, AKT3, A...",353,40
9,9,319,"[ACBD3, ALPP, ALS2, ANK2, ANK3, ANKRD27, ANKRD...",295,24


### Disease Data Sets

In [44]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [45]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [46]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [47]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,1097,"{PITHD1, SAP18, SERP1, NOP16, METTL18, TFB2M, ...",736,361,0.670921
1,1,628,"{IQGAP2, LIMA1, INF2, RHOQ, CORO1C, FSCN1, MAR...",43,585,0.068471
2,2,887,"{COL8A2, SERPINB9, CD86, CTSF, IL2RG, PRF1, DU...",729,158,0.821871
3,3,1036,"{RASGRF1, CRHR1, GPR6, IRX4, TAC4, HAND1, BMP8...",652,384,0.629344
4,4,824,"{HSD17B6, SLC17A3, SREBF1, CYP7A1, LIPA, LCAT,...",403,421,0.489078
5,5,696,"{H2AC12, FBXO5, PRDM9, SASS6, NCAPG, PMS2, SMY...",587,109,0.843391
6,6,508,"{DYNLT5, ARMC2, CFAP53, CLUAP1, TEKT3, WDR54, ...",138,370,0.271654
7,7,323,{},0,323,0.000000
8,8,393,"{ETS1, LYN, PTPN11, DUSP16, ARHGEF3, TRIO, SH2...",389,4,0.989822
9,9,319,"{SNAP23, COG2, VAMP4, TBC1D24, NAA35, MIA3, PP...",317,2,0.993730


In [48]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [49]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1097,RNA Binding (GO:0003723),351/1411,6.843116e-143,[binding],GO_Molecular_Function_2023,1.677234e-145,0.0,0.0,7.920093,2640.224035,EIF4A1;TCERG1;RTCA;RRP1;PPAN;HNRNPU;GPATCH4;HN...,0.248760
96,0,1097,Protein K48-linked Ubiquitination (GO:0070936),16/70,3.159683e-05,[cellular process],GO_Biological_Process_2023,9.036131e-07,0.0,0.0,5.166410,71.900224,RNF34;UBE2B;UBE2E3;CUL3;UBE2D3;UBE3A;RNF6;UBE2...,0.228571
97,0,1097,Protein Neddylation (GO:0045116),9/22,3.887523e-05,[cellular process],GO_Biological_Process_2023,1.129988e-06,0.0,0.0,12.019938,164.592658,UBE2F;DCUN1D5;DCUN1D4;DCUN1D1;UBA3;NEDD8;RNF7;...,0.409091
98,0,1097,"Preribosome, Small Subunit Precursor (GO:0030688)",6/10,4.159620e-05,[protein-containing complex],GO_Cellular_Component_2023,4.661644e-06,0.0,0.0,25.983960,318.982791,LTV1;NOP14;RIOK3;TSR1;RIOK2;RIOK1,0.600000
99,0,1097,"mRNA Cis Splicing, Via Spliceosome (GO:0045292)",8/17,4.214994e-05,[cellular process],GO_Biological_Process_2023,1.244935e-06,0.0,0.0,15.422100,209.685458,CLNS1A;DDX23;SRSF1;WBP4;SDE2;HNRNPC;CACTIN;SNR...,0.470588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,11,132,Intermediate Filament (GO:0005882),23/69,6.461525e-33,[cellular anatomical structure],GO_Cellular_Component_2023,8.811171e-34,0.0,0.0,90.926606,6920.594236,KRT71;KRT80;KRT3;KRT2;KRT1;KRT79;KRT78;KRT10;K...,0.333333
1994,11,132,Supramolecular Fiber Organization (GO:0097435),38/316,1.015771e-35,[cellular process],GO_Biological_Process_2023,1.953405e-37,0.0,0.0,28.486913,2407.886896,KRT80;KRT24;TCHH;KRT20;KRT86;KRT85;KRT40;KRT84...,0.120253
1993,11,132,Keratin Filament (GO:0045095),22/39,8.680165e-38,[cellular anatomical structure],GO_Cellular_Component_2023,7.891059e-39,0.0,0.0,233.541176,20489.755737,KRT71;KRT80;KRT3;KRT2;KRT1;KRT79;KRT78;KRT10;K...,0.564103
1999,11,132,Keratinocyte Differentiation (GO:0030216),11/42,4.221335e-14,"[developmental process, cellular process]",GO_Biological_Process_2023,2.435386e-15,0.0,0.0,58.173021,1957.444847,TGM1;SPRR2F;SPRR3;SPRR2G;CDSN;KRT16;LORICRIN;K...,0.261905


In [50]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [51]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [52]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [53]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [54]:
# twr3

In [55]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [56]:
# terms_with_recurrence

In [57]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [58]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [59]:
# terms_with_rec_merged

In [60]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [61]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [62]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [63]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))